#### DEPENDENCIES IMPORT

## 📖 Guía de uso – Tenerife Travel Guide RAG

Este notebook implementa un **chatbot de viajes sobre Tenerife** que combina:
- **RAG (Retrieval-Augmented Generation)**: responde basándose en el contenido de `TENERIFE.pdf`.
- **Tool-calling**: si la pregunta requiere información de movilidad (cómo ir de un sitio a otro), el modelo puede invocar una herramienta simulada que calcula rutas, tiempos y líneas de guagua.

A continuación se describe **el orden de ejecución de las celdas** y qué hace cada bloque. Ejecuta las celdas de arriba a abajo, en orden, sin saltarte ninguna.

### 1️⃣ Dependencias e instalación (`DEPENDENCIES IMPORT`)
Importa todas las librerías necesarias (LangChain, FAISS, cliente de Gemini, utilidades de limpieza de texto, etc.). No requiere ninguna acción por tu parte.

### 2️⃣ Configuración de la aplicación (`LOAD APP (ENVVARS, TOOL SCHEMAS, INSTATIATION, ETC)`)
Carga las variables de entorno desde el archivo `.env` (API key de Google, modelo de embeddings, modelo de LLM, parámetros de generación, tamaño de chunk, etc.).

- Si alguna variable falta, **el notebook te la pedirá por teclado** (`Enter API key for Google Gemini:`, etc.). Tenlas a mano antes de ejecutar.
- También define el esquema de la herramienta de movilidad (`mobility_tool_schema`) e inicializa el cliente de embeddings y el cliente de Gemini.

> 💡 Si esta celda falla, revisa primero tu archivo `.env` antes de continuar.

### 3️⃣ Carga y limpieza del PDF (`FUNCTION TO LOAD AND CLEANUP DOCUMENT`)
Define `load_cleanup_documents()`, que:
- Carga el PDF `TENERIFE.pdf` página por página.
- Limpia el texto (elimina referencias a imágenes, números de página, bullets raros, espacios sobrantes, etc.).
- Descarta páginas que queden vacías tras la limpieza.

Esta celda **solo define la función**; aún no se ejecuta nada sobre el PDF.

### 4️⃣ Generación del índice vectorial (`FUNCTION TO SAVE DOCUMENT IN VECTORIAL DDBB`)
Aquí ocurre el trabajo pesado de preparación:
1. Se llama a `load_cleanup_documents()` para cargar y limpiar `TENERIFE.pdf`.
2. Se define y ejecuta `save_vectorial_docs()`, que divide el documento en *chunks* y genera sus embeddings con Gemini, guardándolos en un índice **FAISS** (`vector_store`).

> ⏳ Esta celda puede tardar un poco porque genera embeddings para todos los chunks del documento. Es el "cerebro" de búsqueda que usará el chatbot para responder con contexto.

### 5️⃣ Herramienta de movilidad simulada (`TOOLS`)
Define un conjunto de datos simulados (pero realistas) de pueblos de Tenerife, tiempos de viaje y líneas de guagua (TITSA), junto con la función `get_city_mobility()`.

- El LLM puede llamar a esta función cuando el usuario pregunte algo tipo *"¿cómo voy de Puerto de la Cruz a Santa Cruz?"*.
- Devuelve siempre una respuesta estructurada (nunca lanza errores), indicando el mejor transporte, tiempo estimado y punto de recogida.

No requiere acción manual: solo registra la herramienta para que el modelo pueda usarla más adelante.

### 6️⃣ Configuración del LLM con herramientas (`GENERATE LLM CONFIGURATION WITH TOOLS`)
Define las funciones auxiliares que traducen el esquema de la herramienta (`mobility_tool_schema`) al formato que entiende el SDK de Gemini, y construye la configuración final del modelo (temperatura, top-k, top-p, instrucciones de sistema, herramientas disponibles, etc.).

### 7️⃣ Funciones del flujo de conversación (`CHAT EXECUTION FUNCTIONS`)
Define toda la lógica que orquesta una conversación:
- Cómo se recupera contexto relevante del PDF para cada pregunta (RAG).
- Cómo se detecta si el modelo quiere usar la herramienta de movilidad y cómo se ejecuta.
- Cómo se cuentan los tokens consumidos en cada turno.

Es código de soporte: no produce salida visible todavía.

### 8️⃣ Chatbot interactivo (`CHAT BOT MAIN`) — ¡aquí empieza la conversación!
Esta es la celda con la que **interactúas como usuario final**:

1. Al ejecutarla, el notebook te saluda y te pide tu primera pregunta sobre Tenerife (lugares que visitar, rutas, clima, consejos locales, etc.).
2. El modelo responde usando el contenido del PDF como contexto y, si la pregunta lo requiere, consulta la herramienta de movilidad.
3. Tras cada respuesta, se te preguntará si quieres hacer otra pregunta (`yes`/`y` para continuar, cualquier otra cosa para terminar).
4. Al finalizar, verás estadísticas de tokens consumidos y el historial completo de la conversación.

> ⚠️ Esta celda usa `input()`, por lo que requiere ejecución **interactiva** (no funciona en modo "Run All" sin atención, ya que esperará a que escribas tus respuestas).

---

### ✅ Resumen rápido
| Paso | Celda | Qué hace | ¿Necesita tu intervención? |
|------|-------|----------|------------------------------|
| 1 | Dependencias | Importa librerías | No |
| 2 | Configuración | Carga `.env` y credenciales | Solo si faltan variables |
| 3 | Limpieza PDF | Define función de carga/limpieza | No |
| 4 | Índice vectorial | Crea `vector_store` con embeddings del PDF | No (puede tardar) |
| 5 | Herramienta movilidad | Define datos y función de rutas simuladas | No |
| 6 | Configuración LLM | Prepara el modelo con herramientas | No |
| 7 | Funciones de chat | Define lógica RAG + tool-calling | No |
| 8 | Chatbot | Conversación interactiva | **Sí**, responde a los `input()` |


In [36]:
import getpass
import os
import re
from pathlib import Path

from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
# Asegúrate de tener instalado pypdf
from langchain_community.document_loaders import PyPDFLoader

from google import genai
from google.genai import types as genai_types

import logging  # noqa: F401 - reservado para instrumentación futura

import unicodedata

#### LOAD APP (ENVVARS, TOOL SCHEMAS, INSTATIATION, ETC)

In [37]:
load_dotenv()

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")
if not os.environ.get("GOOGLE_EMBEDDING_MODEL"):
    os.environ["GOOGLE_EMBEDDING_MODEL"] = getpass.getpass("Enter Embedding model name: ")
if not os.environ.get("GOOGLE_FAST_MODEL"):
    os.environ["GOOGLE_FAST_MODEL"] = getpass.getpass("Enter Google model name: ")
if not os.environ.get("GOOGLE_LLM_SEED"):
    os.environ["GOOGLE_LLM_SEED"] = getpass.getpass("Enter Google model seed: ")
if not os.environ.get("GOOGLE_LLM_TEMPERATURE"):
    os.environ["GOOGLE_LLM_TEMPERATURE"] = getpass.getpass("Enter Google llm temperature: ")
if not os.environ.get("GOOGLE_LLM_TOPK"):
    os.environ["GOOGLE_LLM_TOPK"] = getpass.getpass("Enter Google llm top k : ")
if not os.environ.get("GOOGLE_LLM_TOPP"):
    os.environ["GOOGLE_LLM_TOPP"] = getpass.getpass("Enter Google llm top p : ")
if not os.environ.get("MAX_OUTPUT_TOKENS_PER_QUESTION"):
    os.environ["MAX_OUTPUT_TOKENS_PER_QUESTION"] = getpass.getpass("Enter Google llm max output tokens per question : ")
if not os.environ.get("GOOGLE_LOGPROBS_ACTIVE"):
    os.environ["GOOGLE_LOGPROBS_ACTIVE"] = getpass.getpass("Enter Google llm log probs (True or False) : ")
if not os.environ.get("CHUNK_SIZE"):
    os.environ["CHUNK_SIZE"] = getpass.getpass("Enter chunk size : ")
if not os.environ.get("CHUNK_OVERLAP"):
    os.environ["CHUNK_OVERLAP"] = getpass.getpass("Enter chunk overlap : ")
if not os.environ.get("CHUNK_NUMBER"):
    os.environ["CHUNK_NUMBER"] = getpass.getpass("Enter chunk number : ")

API_KEY = os.environ["GOOGLE_API_KEY"]
EMBEDDING_MODEL = os.environ["GOOGLE_EMBEDDING_MODEL"]
LLM_MODEL = os.environ["GOOGLE_FAST_MODEL"]
LLM_SEED = int(os.environ["GOOGLE_LLM_SEED"])
LLM_TEMPERATURE = float(os.environ["GOOGLE_LLM_TEMPERATURE"])
LLM_TOPP = float(os.environ["GOOGLE_LLM_TOPP"])
LLM_TOPK = int(os.environ["GOOGLE_LLM_TOPK"])
OUTPUT_TOKENS_PER_QUESTION = int(os.environ["MAX_OUTPUT_TOKENS_PER_QUESTION"])
ACTIVE_LOGPROBS = os.environ["GOOGLE_LOGPROBS_ACTIVE"].strip().lower() == "true"
CHUNK_SIZE = int(os.environ["CHUNK_SIZE"])
CHUNK_OVERLAP = int(os.environ["CHUNK_OVERLAP"])
CHUNK_NUMBER = int(os.environ["CHUNK_NUMBER"])

print("Variables de entorno cargadas correctamente")

mobility_tool_schema = {
    "type": "function",
    "name": "get_city_mobility",
    "description": "Given a origin, destination and, optionally, a transport mode between bus, car, cycle, walk or best, which is chosen by default. Then, it returns the best transportation mode with best ETA and nearest town from origin",
    "strict": True,
    "parameters": {
        "type": "object",
        "properties": {
            "origin": {
                "type": "string",
                "description": "Name of town or city to start visiting destination place. The town or city must be located in Tenerife",
            },
            "destination": {
                "type": "string",
                "description": "Name of place to arrive. The town or city must be located in Tenerife",
            },
            "transport_mode": {
                "type": "string",
                "description": "Mode of transport to go to destination. If transport_mode is not indicated, 'best' will be chosen compare all kind of transportation modes and choose the fastest one",
                "default": "best",
            },
        },
        "required": ["origin", "destination"],
    },
}

print("Esquemas de herramientas cargadas")

google_embedding = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL, api_key=API_KEY)
llm_client = genai.Client()


print("Algortimo embedding y cliente Gemini instanciado correctamente")

model_info = llm_client.models.get(model=LLM_MODEL)
MODEL_MAX_OUTPUT_TOKENS = model_info.output_token_limit

Variables de entorno cargadas correctamente
Esquemas de herramientas cargadas
Algortimo embedding y cliente Gemini instanciado correctamente


#### FUNCTION TO LOAD AND CLEANUP DOCUMENT

In [38]:
def load_cleanup_documents(pdf_path: Path) -> list:
    """Carga un PDF y limpia el contenido textual de cada página.

    Lee el PDF con `PyPDFLoader` y aplica una serie de sustituciones por
    regex para eliminar ruido típico de documentos exportados (referencias
    a imágenes entre corchetes/paréntesis, bullets y caracteres extraños,
    numeración de página, líneas en blanco y espacios redundantes).
    Las páginas que quedan vacías tras la limpieza se descartan.

    Args:
        pdf_path: Ruta al archivo PDF a cargar.

    Returns:
        list[Document]: Documentos de LangChain (una entrada por página),
        con `page_content` limpio.

    Raises:
        FileNotFoundError: Si `pdf_path` no existe.
    """
    if not pdf_path.exists():
        raise FileNotFoundError(f"No se encontró el PDF en {pdf_path.resolve()}")

    loader = PyPDFLoader(str(pdf_path))
    docs = loader.load()

    print("Documento cargado correctamente")

    # Limpiamos mediante regex caracteres extraños asociados a las imagenes
    for doc in docs:
        doc.page_content = re.sub(r'\[.*?\]', '', doc.page_content)  # Elimina [cualquier cosa]
        doc.page_content = re.sub(r'\(.*?\)', '', doc.page_content)  # Elimina (cualquier cosa)
        doc.page_content = re.sub(r'^\s+$', '', doc.page_content, flags=re.MULTILINE)  # Líneas solo con espacios
        doc.page_content = re.sub(r'', '', doc.page_content)  # Bullet points raros (•)
        doc.page_content = re.sub(r'^Página \d+.*$', '', doc.page_content, flags=re.MULTILINE)  # "Página 1 de 10"
        doc.page_content = re.sub(r'�', '', doc.page_content)  # Eliminar carácteres extraños
        doc.page_content = re.sub(r'\n{3,}', '\n\n', doc.page_content)  # Múltiples multilinea
        doc.page_content = re.sub(r' {3,}', ' ', doc.page_content)  # Espacios excesivos → uno
        doc.page_content = doc.page_content.strip()  # Eliminar espacios al inicio y final

    # Eliminación de páginas en blanco tras la limpieza
    docs = [doc for doc in docs if doc.page_content.strip()]

    print(f"Documentos cargados: {len(docs)} (una entrada por página).")
    print(f"Longitud de la primera página: {len(docs[0].page_content)} caracteres")
    print("Muestra del contenido de la primera página:")
    for i, page in enumerate(docs):
        print(f"Page {i} - content length: {len(page.page_content)}")

    return docs

#### FUNCTION TO SAVE DOCUMENT IN VECTORIAL DDBB

In [39]:
doc_list = load_cleanup_documents(Path("../../data/TENERIFE.pdf"))


def save_vectorial_docs(docs: list) -> FAISS:
    """Divide documentos en chunks y los indexa en un vector store FAISS.

    Aplica un `RecursiveCharacterTextSplitter` con `CHUNK_SIZE` y
    `CHUNK_OVERLAP`, conservando el índice de inicio de cada chunk
    (`add_start_index=True`), y genera los embeddings con
    `google_embedding` para construir el índice FAISS.

    Args:
        docs: Lista de objetos `Document` (LangChain) a indexar.

    Returns:
        FAISS: Vector store con los embeddings de los chunks generados.
    """
    splitter_text = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        add_start_index=True
    )

    splitter_text_list = splitter_text.split_documents(docs)
    print(f"Number of splitters: {len(splitter_text_list)}")

    # Generate embeddings and save it as vectos with FAISS
    return FAISS.from_documents(
        documents=splitter_text_list,
        embedding=google_embedding
    )


vector_store = save_vectorial_docs(doc_list)

Documento cargado correctamente
Documentos cargados: 24 (una entrada por página).
Longitud de la primera página: 816 caracteres
Muestra del contenido de la primera página:
Page 0 - content length: 816
Page 1 - content length: 44
Page 2 - content length: 464
Page 3 - content length: 354
Page 4 - content length: 807
Page 5 - content length: 112
Page 6 - content length: 407
Page 7 - content length: 1155
Page 8 - content length: 201
Page 9 - content length: 1208
Page 10 - content length: 1626
Page 11 - content length: 162
Page 12 - content length: 345
Page 13 - content length: 884
Page 14 - content length: 713
Page 15 - content length: 530
Page 16 - content length: 332
Page 17 - content length: 123
Page 18 - content length: 92
Page 19 - content length: 1010
Page 20 - content length: 503
Page 21 - content length: 385
Page 22 - content length: 2187
Page 23 - content length: 474
Number of splitters: 34


#### TOOLS
Método que simula una llamada a una API de movilidad en Tenerife. Dado un origen y un destino
(ciudades reales extraídas de `TENERIFE.pdf`) y, opcionalmente, un modo de transporte
preferido (`bus`, `car`, `cycle`, `walk`) o `best` para comparar todos, devuelve el mejor
transporte, el tiempo estimado y la línea de guagua / punto de recogida en el pueblo más
cercano al origen.

Diseñado para ser invocado por un LLM como *function-calling tool*: nunca lanza excepciones y
siempre devuelve un `dict` serializable a JSON. El cableado con el LLM queda fuera de este
método (lo implementa el usuario).

In [40]:
# --- Simulated Tenerife mobility dataset (real towns from TENERIFE.pdf, invented but plausible data) ---

VALID_TRANSPORT_MODES = ("bus", "car", "cycle", "walk")

# Ciudad/POI -> nombre legible + pueblo más cercano donde se coge el transporte
CITIES = {
    "santa_cruz": {"display": "Santa Cruz de Tenerife", "nearest_town": "Santa Cruz de Tenerife"},
    "la_laguna": {"display": "La Laguna", "nearest_town": "La Laguna"},
    "puerto_de_la_cruz": {"display": "Puerto de la Cruz", "nearest_town": "Puerto de la Cruz"},
    "la_orotava": {"display": "La Orotava", "nearest_town": "La Orotava"},
    "santa_ursula": {"display": "Santa Úrsula", "nearest_town": "Santa Úrsula"},
    "icod": {"display": "Icod de los Vinos", "nearest_town": "Icod de los Vinos"},
    "garachico": {"display": "Garachico", "nearest_town": "Garachico"},
    "buenavista": {"display": "Buenavista del Norte", "nearest_town": "Buenavista del Norte"},
    "masca": {"display": "Masca", "nearest_town": "Buenavista del Norte"},
    "los_gigantes": {"display": "Los Gigantes", "nearest_town": "Santiago del Teide"},
    "adeje": {"display": "Adeje", "nearest_town": "Adeje"},
    "los_cristianos": {"display": "Los Cristianos", "nearest_town": "Arona"},
    "el_teide": {"display": "El Teide", "nearest_town": "La Orotava"},
    "punta_de_teno": {"display": "Punta de Teno", "nearest_town": "Buenavista del Norte"},
    "anaga": {"display": "Anaga", "nearest_town": "La Laguna"},
}

# Alias frecuentes -> clave canónica (formas cortas/largas que un usuario o LLM puede usar)
CITY_ALIASES = {
    "teide": "el_teide",
    "el teide": "el_teide",
    "laguna": "la_laguna",
    "san cristobal de la laguna": "la_laguna",
    "puerto": "puerto_de_la_cruz",
    "puerto cruz": "puerto_de_la_cruz",
    "orotava": "la_orotava",
    "santa ursula": "santa_ursula",
    "icod": "icod",
    "icod de los vinos": "icod",
    "buenavista": "buenavista",
    "buenavista del norte": "buenavista",
    "cristianos": "los_cristianos",
    "gigantes": "los_gigantes",
    "santa cruz": "santa_cruz",
    "santa cruz de tenerife": "santa_cruz",
    "punta de teno": "punta_de_teno",
    "teno": "punta_de_teno",
}

# Tiempo estimado en minutos por modo para cada par (no dirigido). None = modo no viable.
TRAVEL_TIMES = {
    frozenset({"puerto_de_la_cruz", "santa_cruz"}): {"bus": 55, "car": 35, "cycle": 150, "walk": None},
    frozenset({"puerto_de_la_cruz", "la_orotava"}): {"bus": 20, "car": 12, "cycle": 35, "walk": 75},
    frozenset({"puerto_de_la_cruz", "la_laguna"}): {"bus": 50, "car": 30, "cycle": 140, "walk": None},
    frozenset({"la_laguna", "santa_cruz"}): {"bus": 25, "car": 15, "cycle": 45, "walk": 110},
    frozenset({"la_orotava", "santa_ursula"}): {"bus": 18, "car": 10, "cycle": 30, "walk": 70},
    frozenset({"la_orotava", "el_teide"}): {"bus": None, "car": 55, "cycle": 180, "walk": None},
    frozenset({"icod", "garachico"}): {"bus": 15, "car": 9, "cycle": 25, "walk": 60},
    frozenset({"garachico", "buenavista"}): {"bus": 25, "car": 16, "cycle": 40, "walk": None},
    frozenset({"buenavista", "masca"}): {"bus": 35, "car": 25, "cycle": 80, "walk": None},
    frozenset({"buenavista", "punta_de_teno"}): {"bus": 30, "car": 20, "cycle": 45, "walk": None},
    frozenset({"adeje", "los_cristianos"}): {"bus": 20, "car": 12, "cycle": 35, "walk": 80},
    frozenset({"adeje", "los_gigantes"}): {"bus": 40, "car": 25, "cycle": 70, "walk": None},
    frozenset({"santa_cruz", "adeje"}): {"bus": 75, "car": 50, "cycle": None, "walk": None},
    frozenset({"la_laguna", "anaga"}): {"bus": 45, "car": 30, "cycle": 90, "walk": None},
}

# Línea de guagua (TITSA) y punto de recogida por par no dirigido (solo donde el bus es viable).
BUS_LINES = {
    frozenset({"puerto_de_la_cruz", "santa_cruz"}): {"line": "TITSA 103", "stop": "Estación de guaguas de Puerto de la Cruz"},
    frozenset({"puerto_de_la_cruz", "la_orotava"}): {"line": "TITSA 350", "stop": "Estación de guaguas de Puerto de la Cruz"},
    frozenset({"puerto_de_la_cruz", "la_laguna"}): {"line": "TITSA 102", "stop": "Estación de guaguas de Puerto de la Cruz"},
    frozenset({"la_laguna", "santa_cruz"}): {"line": "TITSA 015", "stop": "Intercambiador de La Laguna"},
    frozenset({"la_orotava", "santa_ursula"}): {"line": "TITSA 345", "stop": "Parada Plaza del Ayuntamiento, La Orotava"},
    frozenset({"icod", "garachico"}): {"line": "TITSA 363", "stop": "Estación de guaguas de Icod de los Vinos"},
    frozenset({"garachico", "buenavista"}): {"line": "TITSA 363", "stop": "Parada principal de Garachico"},
    frozenset({"buenavista", "masca"}): {"line": "TITSA 355", "stop": "Estación de Buenavista del Norte"},
    frozenset({"buenavista", "punta_de_teno"}): {"line": "TITSA 369", "stop": "Estación de Buenavista del Norte"},
    frozenset({"adeje", "los_cristianos"}): {"line": "TITSA 467", "stop": "Estación de guaguas de Adeje"},
    frozenset({"adeje", "los_gigantes"}): {"line": "TITSA 473", "stop": "Estación de guaguas de Adeje"},
    frozenset({"santa_cruz", "adeje"}): {"line": "TITSA 110", "stop": "Intercambiador de Santa Cruz"},
    frozenset({"la_laguna", "anaga"}): {"line": "TITSA 077", "stop": "Intercambiador de La Laguna"},
}


def _normalize_city(name):
    """Normaliza un nombre de ciudad a su clave canónica (sin acentos, minúsculas, alias)."""
    text = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode("ascii")
    text = text.strip().lower()
    text = re.sub(r"\s+", " ", text)
    if text in CITY_ALIASES:
        return CITY_ALIASES[text]
    key = text.replace(" ", "_")
    return key


def _error(error_code, message, **extra):
    """Construye una respuesta de error homogénea (siempre serializable a JSON)."""
    payload = {"status": "ERROR", "error_code": error_code, "message": message}
    payload.update(extra)
    return payload


def get_city_mobility(origin, destination, transport_mode="best"):
    """Simula una API de movilidad de Tenerife.

    Dado un `origin` y un `destination` (pueblos/POIs de Tenerife) devuelve el mejor
    transporte para llegar, el tiempo estimado en minutos y la línea de guagua / punto de
    recogida en el pueblo más cercano al origen.

    Args:
        origin: Pueblo de origen en Tenerife (p.ej. "Puerto de la Cruz").
        destination: Pueblo o punto de interés de destino (p.ej. "Santa Cruz", "El Teide").
        transport_mode: Modo preferido ("bus", "car", "cycle", "walk") o "best" para
            comparar todos y recomendar el más rápido. Por defecto "best".

    Returns:
        dict serializable a JSON. En éxito: status="OK" con recommended_transport,
        estimated_minutes, nearest_town, bus_line, pickup_point y all_options. En error:
        status="ERROR" con error_code en {BAD_REQUEST, CITY_NOT_FOUND, INTERNAL_SERVER_ERROR}.
    """
    try:
        # --- BAD_REQUEST ---
        if not isinstance(origin, str) or not origin.strip():
            return _error("BAD_REQUEST", "El parámetro 'origin' es obligatorio y no puede estar vacío.")
        if not isinstance(destination, str) or not destination.strip():
            return _error("BAD_REQUEST", "El parámetro 'destination' es obligatorio y no puede estar vacío.")

        mode = (transport_mode or "best").strip().lower()
        if mode not in VALID_TRANSPORT_MODES + ("best",):
            return _error(
                "BAD_REQUEST",
                f"transport_mode '{transport_mode}' no válido. Usa uno de "
                f"{list(VALID_TRANSPORT_MODES) + ['best']}.",
                valid_modes=list(VALID_TRANSPORT_MODES) + ["best"],
            )

        origin_key = _normalize_city(origin)
        dest_key = _normalize_city(destination)

        if origin_key == dest_key:
            return _error("BAD_REQUEST", "El origen y el destino no pueden ser el mismo lugar.")

        # --- CITY_NOT_FOUND ---
        valid_display = [c["display"] for c in CITIES.values()]
        if origin_key not in CITIES:
            return _error("CITY_NOT_FOUND", f"Origen '{origin}' no encontrado en Tenerife.",
                          valid_cities=valid_display)
        if dest_key not in CITIES:
            return _error("CITY_NOT_FOUND", f"Destino '{destination}' no encontrado en Tenerife.",
                          valid_cities=valid_display)

        # --- INTERNAL_SERVER_ERROR: par sin datos de ruta (fallo simulado del backend) ---
        pair = frozenset({origin_key, dest_key})
        if pair not in TRAVEL_TIMES:
            return _error(
                "INTERNAL_SERVER_ERROR",
                f"No se pudo calcular la ruta entre '{CITIES[origin_key]['display']}' y "
                f"'{CITIES[dest_key]['display']}'. Inténtalo de nuevo más tarde.",
            )

        times = TRAVEL_TIMES[pair]
        viable = {m: t for m, t in times.items() if t is not None}
        if not viable:
            return _error(
                "INTERNAL_SERVER_ERROR",
                "No hay ningún modo de transporte disponible para esta ruta.",
            )

        # --- Selección de transporte ---
        best_mode = min(viable, key=viable.get)
        note = None
        if mode == "best":
            chosen = best_mode
        elif mode in viable:
            chosen = mode
        else:
            # Modo solicitado no viable: caemos al mejor disponible y avisamos.
            chosen = best_mode
            note = (f"El modo '{mode}' no está disponible para esta ruta; "
                    f"se recomienda '{best_mode}' en su lugar.")

        bus_info = BUS_LINES.get(pair, {})
        result = {
            "status": "OK",
            "origin": CITIES[origin_key]["display"],
            "destination": CITIES[dest_key]["display"],
            "recommended_transport": chosen,
            "estimated_minutes": viable[chosen],
            "nearest_town": CITIES[origin_key]["nearest_town"],
            "bus_line": bus_info.get("line"),
            "pickup_point": bus_info.get("stop"),
            "all_options": viable,
        }
        if note:
            result["note"] = note
        return result

    except Exception as exc:  # noqa: BLE001 - frontera de la API simulada: nunca propagar
        return _error("INTERNAL_SERVER_ERROR", f"Error inesperado en el servicio de movilidad: {exc}")


print("get_city_mobility lista. Ciudades disponibles:")
print(", ".join(c["display"] for c in CITIES.values()))

get_city_mobility lista. Ciudades disponibles:
Santa Cruz de Tenerife, La Laguna, Puerto de la Cruz, La Orotava, Santa Úrsula, Icod de los Vinos, Garachico, Buenavista del Norte, Masca, Los Gigantes, Adeje, Los Cristianos, El Teide, Punta de Teno, Anaga


#### GENERATE LLM CONFIGURATION WITH TOOLS

In [41]:
def parse_schema(parameters: dict) -> genai_types.Schema:
    """Convierte un esquema JSON (estilo OpenAI function-calling) a `genai_types.Schema`.

    Recorre recursivamente las propiedades de un esquema de tipo "object"
    para construir el equivalente en el SDK de Gemini, preservando
    descripciones y campos `required`.

    Args:
        parameters: Diccionario con el esquema JSON de los parámetros de la
            herramienta (claves `type`, `description`, `properties`,
            `required`).

    Returns:
        genai_types.Schema: Esquema equivalente listo para usarse en una
        `FunctionDeclaration` de Gemini.
    """
    type_map = {
        "string": genai_types.Type.STRING,
        "object": genai_types.Type.OBJECT,
    }

    schema_type = type_map[parameters["type"]]
    is_object = schema_type == genai_types.Type.OBJECT

    # Only parse properties for object types.
    properties = {}
    if is_object and "properties" in parameters:
        for key, value in parameters["properties"].items():
            properties[key] = parse_schema(value)

    return genai_types.Schema(
        type=schema_type,
        description=parameters.get("description", ""),
        properties=properties if is_object else None,
        required=parameters.get("required", []) if is_object else None,
        enum=None,
    )


def generate_tools(schema_list: list):
    tool_information = dict()
    
    for schema in schema_list:
        function_name = schema.get("name")
        schema_parsed = genai_types.FunctionDeclaration(
            name=function_name,
            description=schema.get("description"),
            parameters=parse_schema(schema.get("parameters")),
        )

        tool_information[function_name] = genai_types.Tool(function_declarations=[schema_parsed])
    
    return tool_information

def generate_configuration(schema_list: list):

    tools = generate_tools(schema_list)

    with open("../../data/templates/system_instructions.txt", "r", encoding="utf-8") as f:
        instruction_content = f.read()
        model_config = genai_types.GenerateContentConfig(
            system_instruction=instruction_content,
            response_modalities=["TEXT"],
            temperature=LLM_TEMPERATURE,
            top_k=LLM_TOPK,
            top_p=LLM_TOPP,
            seed=LLM_SEED,
            max_output_tokens=OUTPUT_TOKENS_PER_QUESTION,
            response_logprobs=ACTIVE_LOGPROBS,
            thinking_config=genai_types.GenerationConfigThinkingConfig(
               thinking_level=genai_types.ThinkingLevel.MEDIUM
            ),
            tools=list(tools.values()),
            tool_config=genai_types.ToolConfig(
               function_calling_config=genai_types.FunctionCallingConfig(
                   mode=genai_types.FunctionCallingConfigMode.AUTO
                )
            )
        )
    
    print("Configuración establecida correctamente")

    return model_config

#### CHAT EXECUTION FUNCTIONS

In [42]:
def _format_docs_with_sources(documents):
    """Formatea documentos recuperados en un bloque de texto con metadatos de origen.

    Construye, para cada documento, una cabecera con la fuente, el número de
    página y el identificador de chunk, seguida de su contenido. El resultado
    está pensado para inyectarse como contexto en el prompt del LLM.

    Args:
        documents: Lista de objetos `Document` (LangChain) recuperados del
            vector store, cada uno con `page_content` y `metadata` opcionales
            (`source_name`, `page`, `chunk_id`).

    Returns:
        str: Concatenación de los bloques formateados, uno por documento.
    """
    blocks = []
    for i, doc in enumerate(documents, start=1):
        source = doc.metadata.get("source_name", "fuente desconocida")
        page = doc.metadata.get("page", "?")
        chunk_id = doc.metadata.get("chunk_id", "?")
        blocks.append(
            f"""[Fuente {i}: {source}, página {page}, chunk {chunk_id}]
                {doc.page_content}
            """
        )
    return "".join(blocks)


def _generate_input_message_with_rag(user_message):
    """Construye el mensaje de entrada para el LLM enriquecido con contexto RAG.

    Recupera del vector store los `CHUNK_NUMBER` documentos más similares al
    mensaje del usuario, los formatea con `_format_docs_with_sources` y los
    combina con la pregunta original en un único string.

    Args:
        user_message: Pregunta o mensaje del usuario en lenguaje natural.

    Returns:
        str: Mensaje combinado con el formato
        `"context: {context}; user question: {user_message}"`.
    """
    retrieved_docs = vector_store.similarity_search(user_message, CHUNK_NUMBER)
    context = _format_docs_with_sources(retrieved_docs)

    return f"context: {context}; user question: {user_message}"


def tool_usage_detection(llm_response):
    """Detecta y ejecuta la herramienta solicitada por el LLM en su respuesta.

    Inspecciona `llm_response.function_calls`: si el modelo no solicita
    ninguna herramienta, no hace nada más. Si solicita `get_city_mobility`,
    extrae sus argumentos y la invoca. Cualquier otra herramienta solicitada
    se considera no soportada.

    Args:
        llm_response: Objeto de respuesta del LLM (Gemini) que puede contener
            `function_calls` con las herramientas solicitadas por el modelo.

    Returns:
        tuple[str | None, dict | None]: Una tupla `(tool_name, tool_response)`.
        Ambos valores son `None` si el modelo no solicitó ninguna herramienta.

    Raises:
        ValueError: Si el modelo solicita una herramienta distinta de
            `get_city_mobility`.
    """
    if not llm_response.function_calls:
        return (None, None)

    call = llm_response.function_calls[0]
    args = dict(call.args)

    if call.name != "get_city_mobility":
        raise ValueError(f"Herramienta no soportada: {call.name}")

    tool_result = get_city_mobility(**args)
    return (call.name, tool_result)


def average_output_tokens(results):
    """Calcula el promedio de tokens de salida consumidos por una lista de turnos.

    Útil para estimar un valor razonable de `max_output_tokens` en base al
    consumo real observado en un conjunto de preguntas de prueba.

    Args:
        results: Lista de diccionarios de turno (los devueltos por
            `run_chat_conversation`), cada uno con la clave `output_tokens`.

    Returns:
        dict: Estadísticas con las claves `avg`, `min`, `max` y `count`.
    """
    token_counts = [r["output_tokens"] for r in results if r.get("output_tokens") is not None]

    if not token_counts:
        return {"avg": 0, "min": 0, "max": 0, "count": 0}

    return {
        "avg": sum(token_counts) / len(token_counts),
        "min": min(token_counts),
        "max": max(token_counts),
        "count": len(token_counts),
    }


def run_chat_conversation(chat, user_message):
    """Ejecuta un turno completo de conversación con soporte RAG y tool-calling.

    Genera el mensaje de entrada enriquecido con contexto RAG, lo envía al
    chat y comprueba si el modelo solicita el uso de una herramienta. Si es
    así, ejecuta la herramienta (`tool_usage_detection`) y envía su resultado
    de vuelta al modelo como `FunctionResponse` para obtener la respuesta final.

    Args:
        chat: Sesión de chat de `genai.Client` (`llm_client.chats.create`).
        user_message: Pregunta del usuario en lenguaje natural.
        output_tokens_remain: Límite de tokens de salida disponible para esta
            llamada, usado como `max_output_tokens` en la configuración.

    Returns:
        dict: Diccionario con las claves `question`, `answer`, `prompt`,
        `tool_name_used` (o `"none"` si no se usó ninguna herramienta) y
        `output_tokens` (tokens de salida consumidos en la respuesta final).
    """
    input_message = _generate_input_message_with_rag(user_message)

    llm_response = chat.send_message(message=input_message)
    tool_name, tool_response = tool_usage_detection(llm_response)

    if None not in (tool_name, tool_response):
        llm_response = chat.send_message(
            config=[
                genai_types.Part(
                    function_response=genai_types.FunctionResponse(
                        name=tool_name,
                        response={"result": tool_response},
                    )
                ),
            ]
        )

    return {
        "question": user_message,
        "answer": llm_response.text,
        "prompt": input_message,
        "tool_name_used": "none" if tool_name is None else tool_name,
        "output_tokens": llm_response.usage_metadata.candidates_token_count,
    }

#### CHAT BOT MAIN

In [49]:
output_tokens_remain = MODEL_MAX_OUTPUT_TOKENS

chat = llm_client.chats.create(
    model=LLM_MODEL,
    config=generate_configuration([mobility_tool_schema])
)

print("Welcome to Raul's Chatbot")
print(
    "I'm your Tenerife travel guide. Ask me anything about the island: "
    "places to visit, routes, weather, local tips and more."
)

user_message = input("What would you like to know about Tenerife? ")
information_history = []

while True:
    response_information = run_chat_conversation(chat, user_message)
    information_history.append(response_information)

    output_tokens_consumed = response_information["output_tokens"] or 0
    output_tokens_remain -= output_tokens_consumed

    print(f"\nYou: {response_information['question']}")
    print(f"Bot: {response_information['answer']}")
    print(f"[Output tokens consumed: {output_tokens_consumed} | Remaining: {output_tokens_remain} / {MODEL_MAX_OUTPUT_TOKENS}]")

    if output_tokens_remain <= 0:
        print("\nSe ha alcanzado el límite de tokens de salida disponibles. Finalizando la conversación.")
        break

    ask_more = input("\nWould you like to ask anything else about Tenerife? (yes/no) ")
    if ask_more.strip().lower() not in ("yes", "y"):
        break

    user_message = input("What's your next question? ")

print("\n" + "==" * 30)
print("CONVERSATION HISTORY")
for message in chat.get_history():
    print(f"\n[{message.role.title()}]", end=": ")
    print(message.parts[0].text)


Configuración establecida correctamente
Welcome to Raul's Chatbot
I'm your Tenerife travel guide. Ask me anything about the island: places to visit, routes, weather, local tips and more.


TypeError: Chat.send_message() missing 1 required positional argument: 'message'